<a id="gh200-gpu"></a>
# 01-2. GH200 — GPU 메모리 경로와 프로파일링

**세션:** 14:00–14:30  
**목표:** 같은 벡터 덧셈을 세 가지 메모리 방식으로 실행하고, 데이터 크기·통합 메모리 이동 정책·프로파일 대상을 직접 바꿔 실행 기록을 해석합니다.

이 노트북은 **수정 → 실행 → 비교 → 확인** 순서로 진행합니다. 계산 노드에서는 통합 SIF의 `nvcc`, Nsight Systems와 nvbandwidth만 사용합니다.


## 강의자료에서 코드로

- **CUDA 통합 메모리:** `cudaMallocManaged`는 CPU와 GPU가 같은 포인터를 사용하게 하며, 데이터는 요구 시 접근되거나 `cudaMemPrefetchAsync`로 미리 옮길 수 있습니다.
- **세 가지 메모리 방식:** 명시적 복사, CUDA 통합 메모리, 시스템 할당 메모리는 할당·접근 방식이 다릅니다. GH200의 지원 구성에서는 ATS를 통해 일반 시스템 메모리에 접근할 수 있으며, 아래 프로그램이 실제 지원 여부를 판정합니다.
- **컴파일러 메모리 모드와 Runtime API:** NVHPC의 메모리 모드는 컴파일러 수준의 추상화입니다. 이번 실습은 할당과 이동이 코드에 드러나도록 CUDA Runtime API를 직접 사용합니다.
- **Nsight Systems:** 계산 노드에서 `nsys profile`로 기록을 수집하고, CUDA API·GPU 커널·메모리 활동의 집계와 타임라인을 해석합니다.
- **nvbandwidth:** 애플리케이션 전체 시간이 아니라 선택한 메모리 복사 경로의 전송 대역폭을 측정합니다.

강의자료의 개념을 아래 세 소스의 실제 API와 연결한 뒤, 선택한 경로를 직접 프로파일링합니다.


## 1. Hopper GPU와 프로파일링 도구 확인 — 2분

도구가 없거나 GPU가 GH200으로 표시되지 않으면 설치를 시도하지 말고 강사에게 알립니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "gh200" / "notebook_utils.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("labs/gh200/notebook_utils.py를 찾지 못했습니다.")

LAB_DIR = REPO_ROOT / "labs" / "gh200"
CUDA_SOURCE_DIR = LAB_DIR / "cuda_memory"
WORK_DIR = REPO_ROOT / "work" / "gh200"
BIN_DIR = WORK_DIR / "bin"
PROFILE_DIR = WORK_DIR / "profiles"
BIN_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from notebook_utils import print_tool_status, read_image_manifest, run, system_summary

required_tools = ("nvcc", "nsys", "nvbandwidth")
if not print_tool_status(required_tools):
    raise EnvironmentError("필수 도구가 없습니다. 계산 노드에서 설치하지 말고 강사에게 알리세요.")

summary = system_summary()
for key, value in summary.items():
    print(f"{key:14s}: {value}")
if str(summary["architecture"]).lower() not in {"aarch64", "arm64"}:
    raise EnvironmentError(f"ARM64 환경이 아닙니다: {summary['architecture']}")
if "GH200" not in str(summary.get("gpu", "")).upper():
    raise EnvironmentError(f"GH200을 확인하지 못했습니다: {summary.get('gpu')}")
if read_image_manifest() is None:
    raise FileNotFoundError("통합 이미지 manifest를 찾지 못했습니다: /etc/ksc2026-image.json")
print("이미지 구성 정보: PASS")


## 2. 세 가지 메모리 방식의 코드를 대조하기 — 3분

| 방식 | 소스에서 찾을 API | 데이터가 GPU에 도달하는 방식 |
|---|---|---|
| 명시적 복사 | `cudaMalloc`, `cudaMemcpy` | 호스트·디바이스 메모리를 따로 만들고 코드가 복사 시점을 지정 |
| CUDA 통합 메모리 | `cudaMallocManaged`, 선택적 `cudaMemPrefetchAsync` | 같은 포인터를 사용하며 런타임이 접근·배치를 관리 |
| 시스템 할당 메모리 | `new`, CUDA 런타임 속성 조회 | 지원되는 시스템에서 GPU가 일반 Linux 메모리에 직접 접근 |

시스템 할당 메모리는 세 번째 **할당 방식**입니다. ATS(Address Translation Service)와 HMM(Heterogeneous Memory Management)은 GPU가 이 메모리에 접근할 때 사용할 수 있는 **주소 변환·일관성 경로**입니다. 프로그램은 `cudaDevAttrPageableMemoryAccess`와 `cudaDevAttrPageableMemoryAccessUsesHostPageTables`를 읽어 지원 여부와 실제 경로를 판정합니다. 지원되지 않으면 `SKIP`을 출력하며, 이 결과를 다른 방식의 성능값으로 대신 추정하지 않습니다.


In [ ]:
memory_sources = {
    "explicit": CUDA_SOURCE_DIR / "explicit.cu",
    "managed": CUDA_SOURCE_DIR / "managed.cu",
    # 파일명은 hmm.cu이지만 실행 결과가 ATS와 HMM을 구분합니다.
    "system": CUDA_SOURCE_DIR / "hmm.cu",
}
for source in memory_sources.values():
    if not source.is_file():
        raise FileNotFoundError(source)

def show_api_lines(label, path, markers):
    print(f"\n[{label}: {path.relative_to(REPO_ROOT)}]")
    for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if any(marker in line for marker in markers):
            print(f"{number:3d}: {line.strip()}")

show_api_lines("명시적 복사", memory_sources["explicit"], ("cudaMalloc(", "cudaMemcpy("))
show_api_lines("통합 메모리", memory_sources["managed"], ("cudaMallocManaged", "cudaMemPrefetchAsync", "use_prefetch"))
show_api_lines("시스템 할당", memory_sources["system"], ("PageableMemoryAccess", "new float", "coherency"))


## 3. SM90 대상으로 CUDA 소스 빌드 — 3분

`-arch=sm_90`은 GH200의 Hopper GPU용 코드를 생성합니다. 실제 터미널 명령은 아래 형태입니다. `!`는 Jupyter 셀에서 셸 명령을 실행한다는 뜻이며, 다음 셀은 세 명령을 반복 실행하고 종료 코드까지 확인합니다.

```bash
!nvcc -O3 -std=c++17 -arch=sm_90 -Xcompiler -fopenmp \
      labs/gh200/cuda_memory/explicit.cu -o work/gh200/bin/cuda-explicit
```

`-Xcompiler -fopenmp`는 호스트 쪽 초기화 루프를 Grace CPU 코어 전체로 병렬화합니다. 뒤에서 HBM보다 큰 배열을 다룰 때 첫 접촉(first touch) 시간이 크게 줄어듭니다.


In [ ]:
for name, source in memory_sources.items():
    output = BIN_DIR / f"cuda-{name}"
    run(
        [
            "nvcc", "-O3", "-std=c++17", "-arch=sm_90",
            "-Xcompiler", "-fopenmp",
            str(source), "-o", str(output),
        ],
        timeout=600,
    )
    if not output.is_file():
        raise FileNotFoundError(output)
print("CUDA 빌드 결과: PASS")


## 4. 데이터 크기와 통합 메모리 정책을 직접 선택 — 4분

`cuda-managed`는 두 실행 모드를 제공합니다.

- `demand`: CPU에서 초기화한 뒤 별도의 사전 이동 요청 없이 GPU 커널이 접근합니다.
- `prefetch`: 커널 실행 전에 `cudaMemPrefetchAsync`로 GPU 배치를 요청합니다.

GH200에서는 요구 시 접근이 반드시 ‘느리다’고 단정할 수 없습니다. 데이터 크기, 접근 패턴, 현재 배치와 런타임 정책에 따라 직접 접근이나 이동이 선택될 수 있으므로 Nsight Systems 기록으로 확인해야 합니다.


In [ ]:
# 참가자 수정 ①: 데이터 크기와 먼저 확인할 통합 메모리 모드를 고르세요.
ELEMENTS = 1 << 24
MANAGED_MODES = ["demand", "prefetch"]

if not 1024 <= ELEMENTS <= (1 << 28):
    raise ValueError("ELEMENTS는 1024 이상 2**28 이하로 설정하세요.")
if not MANAGED_MODES or any(mode not in {"demand", "prefetch"} for mode in MANAGED_MODES):
    raise ValueError("MANAGED_MODES에는 demand와 prefetch만 사용할 수 있습니다.")

print(f"원소 수          : {ELEMENTS:,}")
print(f"배열 하나의 크기 : {ELEMENTS * 4 / 1024**2:.1f} MiB")
print("통합 메모리 모드 :", MANAGED_MODES)


In [ ]:
PROGRAM_COMMANDS = {
    "explicit": [BIN_DIR / "cuda-explicit", str(ELEMENTS)],
    "system": [BIN_DIR / "cuda-system", str(ELEMENTS)],
}
for mode in MANAGED_MODES:
    PROGRAM_COMMANDS[f"managed-{mode}"] = [
        BIN_DIR / "cuda-managed",
        str(ELEMENTS),
        mode,
    ]

MEMORY_RESULTS = {}
for name, command in PROGRAM_COMMANDS.items():
    completed = run(command, timeout=300)
    MEMORY_RESULTS[name] = completed.stdout.strip()
    if "result=PASS" not in completed.stdout and "result=SKIP" not in completed.stdout:
        raise RuntimeError(f"{name} 결과를 확인할 수 없습니다.")

SYSTEM_MEMORY_SUPPORTED = "result=SKIP" not in MEMORY_RESULTS["system"]
if SYSTEM_MEMORY_SUPPORTED:
    system_path = (
        "ATS / hardware-coherent"
        if "ATS/hardware-coherent" in MEMORY_RESULTS["system"]
        else "HMM / software-coherent"
    )
    print(f"\n시스템 할당 메모리 경로: {system_path}")
else:
    print("\n시스템 할당 메모리 경로: SKIP (GPU 직접 접근 기능 미지원)")


## 5. 프로파일할 경로를 고르고 Nsight Systems 실행 — 5분

최소 두 경로를 선택해야 비교할 수 있습니다. 기본값은 **명시적 복사와 통합 메모리(demand)** 두 개입니다. 시간이 남으면 `managed-prefetch`나 `system`을 추가하세요.

실제 명령은 다음 형태입니다. `--trace=cuda,osrt`는 CUDA API와 운영체제 런타임 이벤트를, `--cuda-memory-usage=true`는 CUDA 메모리 사용 정보를 수집합니다. 프로파일러에는 오버헤드가 있으므로 프로그램의 원래 전체 실행 시간과 같은 값으로 해석하지 않습니다.

```bash
!nsys profile --trace=cuda,osrt --sample=none --cuda-memory-usage=true --output=work/gh200/profiles/managed-demand work/gh200/bin/cuda-managed 16777216 demand
```


In [ ]:
# 참가자 수정 ②: 비교할 프로파일 대상을 두 개 이상 고르세요.
PROFILE_TARGETS = ["explicit", "managed-demand"]  # 여유가 있으면 managed-prefetch, system 추가
if SYSTEM_MEMORY_SUPPORTED:
    print("system을 PROFILE_TARGETS에 추가할 수 있습니다.")

available_targets = set(PROGRAM_COMMANDS)
unknown_targets = [name for name in PROFILE_TARGETS if name not in available_targets]
if unknown_targets:
    raise ValueError(f"사용할 수 없는 프로파일 대상입니다: {unknown_targets}")
if "system" in PROFILE_TARGETS and not SYSTEM_MEMORY_SUPPORTED:
    raise ValueError("이 GPU에서는 시스템 할당 메모리 경로가 SKIP이므로 system을 프로파일할 수 없습니다.")
if len(set(PROFILE_TARGETS)) < 2:
    raise ValueError("비교를 위해 서로 다른 프로파일 대상을 두 개 이상 선택하세요.")
print("프로파일 대상:", PROFILE_TARGETS)


In [ ]:
PROFILE_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
NSYS_REPORTS = {}
for name in PROFILE_TARGETS:
    profile_base = PROFILE_DIR / f"{PROFILE_RUN_ID}_{name}_n{ELEMENTS}"
    report_path = profile_base.with_suffix(".nsys-rep")
    run(
        [
            "nsys",
            "profile",
            "--trace=cuda,osrt",
            "--sample=none",
            "--cuda-memory-usage=true",
            "--force-overwrite=true",
            "--output",
            str(profile_base),
            *PROGRAM_COMMANDS[name],
        ],
        timeout=600,
    )
    if not report_path.is_file():
        raise FileNotFoundError(report_path)
    NSYS_REPORTS[name] = report_path
    print(f"\n===== {name}: CUDA API·커널·메모리 활동 집계 =====")
    run(
        [
            "nsys",
            "stats",
            "--report",
            "cuda_api_sum,cuda_gpu_kern_sum,cuda_gpu_mem_time_sum,cuda_gpu_mem_size_sum",
            str(report_path),
        ],
        timeout=600,
    )

print("\n생성한 Nsight Systems 보고서")
for name, path in NSYS_REPORTS.items():
    print(f"{name:18s} {path.relative_to(REPO_ROOT)}")


### 강사와 함께 확인할 출력

- `explicit`의 CUDA API 집계에서 `cudaMemcpy` 호출을 찾습니다.
- `managed-demand`와 `managed-prefetch`의 API 호출 목록을 나란히 비교합니다.
- `.nsys-rep` 타임라인에서 API 호출과 GPU 커널의 실행 순서를 확인합니다.
- 시스템 할당 경로가 `ATS`, `HMM`, `SKIP` 중 무엇으로 판정됐는지 실행 출력을 확인합니다.
- 메모리 방식만 비교할 수 있도록 데이터 크기와 벡터 덧셈 커널이 같은지 확인합니다.

CLI 집계표와 `.nsys-rep` 타임라인을 강사 화면에서 함께 읽습니다.


## 6. HBM보다 큰 데이터 다루기 — 8분

여기까지는 GPU 메모리에 충분히 들어가는 크기였습니다. 이제 **일부러 HBM보다 큰 배열**을 만들어 세 방식이 어떻게 갈리는지 봅니다. 이것이 GH200이 x86+GPU와 가장 크게 달라지는 지점입니다.

| 방식 | HBM보다 큰 데이터에서 |
|---|---|
| 명시적 복사 | `cudaMalloc`이 실패합니다. 데이터를 나눠 여러 번 옮기도록 **코드를 다시 짜야** 합니다 |
| CUDA 통합 메모리 | 할당은 됩니다. HBM에 안 들어가는 부분은 접근할 때 옮기거나 원격으로 읽습니다 |
| 시스템 할당 메모리 | `new`로 잡은 Grace의 LPDDR5X를 GPU가 **그대로 읽습니다**. 옮기지 않습니다 |

x86+GPU에서는 마지막 경로가 PCIe를 통해야 하므로 대역폭이 한 자릿수 GB/s대로 떨어집니다. GH200은 NVLink-C2C로 연결되어 있어 같은 코드가 훨씬 높은 대역폭으로 동작합니다.

> **크기는 코드가 실행 시점에 정합니다.** 이 노드의 HBM 여유와 시스템 메모리 여유를 읽어 안전한 값을 계산합니다. 한 노드를 여러 참가자가 함께 쓰는 경우를 고려해 시스템 메모리의 일부만 사용하며, 여유가 부족하면 `SKIP`으로 건너뜁니다. 값을 임의로 키우지 마세요.


In [ ]:
# 이 노드에서 안전한 "HBM 초과" 크기를 계산합니다.
OVERSUBSCRIBE_RATIO = 1.25   # 두 배열 합이 HBM 여유의 몇 배가 되게 할지
SYSTEM_MEMORY_SHARE = 0.35   # 시스템 메모리 여유 중 사용할 비율 (동시 사용자 고려)

def gpu_free_total_bytes():
    query = run(
        ["nvidia-smi", "--query-gpu=memory.free,memory.total",
         "--format=csv,noheader,nounits"],
        timeout=60,
    )
    free_mib, total_mib = (int(v) for v in query.stdout.splitlines()[0].split(","))
    return free_mib * 1024**2, total_mib * 1024**2

def system_available_bytes():
    for line in Path("/proc/meminfo").read_text(encoding="utf-8").splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("/proc/meminfo에서 MemAvailable을 찾지 못했습니다.")

HBM_FREE, HBM_TOTAL = gpu_free_total_bytes()
SYSTEM_AVAILABLE = system_available_bytes()

target_total = int(HBM_FREE * OVERSUBSCRIBE_RATIO)
budget_total = int(SYSTEM_AVAILABLE * SYSTEM_MEMORY_SHARE)
chosen_total = min(target_total, budget_total)
BIG_ELEMENTS = min(chosen_total // 8, 1 << 36)   # 배열 2개 × float 4바이트
BIG_TOTAL_BYTES = BIG_ELEMENTS * 8

OVERSUBSCRIPTION_READY = BIG_TOTAL_BYTES > HBM_FREE and BIG_ELEMENTS >= 1024

print(f"HBM 전체        : {HBM_TOTAL / 2**30:8.1f} GiB")
print(f"HBM 여유        : {HBM_FREE / 2**30:8.1f} GiB")
print(f"시스템 메모리 여유: {SYSTEM_AVAILABLE / 2**30:8.1f} GiB")
print(f"사용할 예산      : {budget_total / 2**30:8.1f} GiB "
      f"(여유의 {SYSTEM_MEMORY_SHARE:.0%})")
print("-" * 60)
print(f"배열 2개 합계    : {BIG_TOTAL_BYTES / 2**30:8.1f} GiB "
      f"(HBM 여유의 {BIG_TOTAL_BYTES / HBM_FREE:.2f}배)")
print(f"원소 수          : {BIG_ELEMENTS:,}")

if not OVERSUBSCRIPTION_READY:
    print("\nSKIP: 시스템 메모리 여유가 부족해 HBM을 넘기는 크기를 만들 수 없습니다.")
    print("      강사에게 알리고 다음 절로 넘어가세요.")


실행합니다. **명시적 복사는 실패하는 것이 정상입니다.** 실패 방식과 메시지를 확인하세요.

큰 배열의 첫 접촉(first touch)에 시간이 걸리므로 각 실행이 수십 초 걸릴 수 있습니다.

In [ ]:
OVERSUBSCRIPTION_RESULTS = {}

if OVERSUBSCRIPTION_READY:
    big_commands = {
        "explicit": [BIN_DIR / "cuda-explicit", str(BIG_ELEMENTS)],
        "managed-demand": [BIN_DIR / "cuda-managed", str(BIG_ELEMENTS), "demand"],
    }
    if SYSTEM_MEMORY_SUPPORTED:
        big_commands["system"] = [BIN_DIR / "cuda-system", str(BIG_ELEMENTS)]

    for name, command in big_commands.items():
        print(f"\n----- {name} -----")
        started = datetime.now(timezone.utc)
        completed = run(command, timeout=1800)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()
        output = completed.stdout.strip()
        OVERSUBSCRIPTION_RESULTS[name] = {"output": output, "seconds": elapsed}
        print(f"({elapsed:.1f} s)")

    print("\n" + "=" * 60)
    print("HBM 초과 요약")
    for name, record in OVERSUBSCRIPTION_RESULTS.items():
        text = record["output"]
        if "result=PASS" in text:
            verdict = "PASS — HBM보다 큰 데이터를 그대로 처리했습니다"
        elif "result=OOM" in text:
            verdict = "OOM — 디바이스 메모리에 담지 못했습니다 (예상된 결과)"
        elif "result=HOST_OOM" in text:
            verdict = "HOST_OOM — 시스템 메모리도 부족했습니다"
        else:
            verdict = "CHECK — 출력을 확인하세요"
        print(f"  {name:16s} {verdict}")
    print("=" * 60)
else:
    print("SKIP: 앞 셀에서 크기를 정하지 못했습니다.")


### 강사와 함께 확인할 지점

- `explicit`이 `result=OOM`으로 끝났습니까? 출력의 `device_free`와 요청한 `bytes`를 비교해 보세요. **코드를 고치지 않으면 이 데이터는 처리할 수 없습니다.**
- `managed-demand`와 `system`은 같은 크기를 어떻게 처리했습니까? 두 방식의 실행 시간 차이는 어디서 왔을까요?
- `system` 경로에서 GPU는 Grace의 LPDDR5X를 직접 읽었습니다. 이 접근이 지나가는 링크가 다음 절에서 측정할 NVLink-C2C입니다.
- x86+GPU에서 같은 코드를 돌리면 무엇이 달라질까요? (힌트: 세 방식 중 무엇이 남고 무엇이 사라지는지)


## 7. nvbandwidth로 링크 전송 대역폭 확인 — 4분

앞 절에서 GPU가 Grace의 LPDDR5X를 직접 읽었습니다. 그 접근이 지나간 링크가 **NVLink-C2C**입니다. 이제 그 링크가 초당 몇 바이트를 옮기는지 직접 잽니다.

프로그램 전체 시간과는 다른 지표입니다. 벡터 덧셈의 실행 시간에는 커널·할당·초기화가 섞여 있지만 nvbandwidth는 선택한 복사 경로만 측정합니다.

복사 방식도 두 가지를 함께 봅니다.

- `_ce` — **Copy Engine**. 전용 DMA 엔진이 백그라운드로 옮깁니다. SM은 계산에 그대로 씁니다.
- `_sm` — **SM 커널 복사**. SM이 직접 복사 커널을 돌립니다. 큰 버퍼에서 더 높은 대역폭이 나올 수 있지만 그동안 SM은 계산을 못 합니다.

```bash
!nvbandwidth -t host_to_device_memcpy_ce device_to_host_memcpy_ce \
             host_to_device_memcpy_sm device_to_host_memcpy_sm
```

### 측정값을 무엇과 비교할 것인가

**이 숫자는 다른 플랫폼과 비교해야 의미가 생깁니다.** 아래는 측정값이 아니라 각 링크의 일반적으로 알려진 값입니다. 여러분이 방금 잰 값을 이 표 옆에 놓아 보세요.

| CPU–GPU 연결 | 단방향 이론 대역폭 | 비고 |
|---|---:|---|
| PCIe Gen4 x16 | 약 32 GB/s | 다수의 기존 x86 + GPU 서버 |
| PCIe Gen5 x16 | 약 64 GB/s | 최신 x86 + H100 구성 |
| **NVLink-C2C (GH200)** | **약 450 GB/s** | 양방향 합계 900 GB/s |

PCIe 구간은 실측이 이론값의 80~90% 수준으로 나오는 것이 보통입니다. 자릿수 차이를 보는 것이 목적이므로 정확한 실측 비교보다 **배수**에 주목하세요.

앞 절에서 HBM보다 큰 배열을 GPU가 그대로 읽을 수 있었던 이유가 이 표의 마지막 줄입니다. PCIe 위에서 같은 일을 하면 대역폭이 한 자릿수 배로 떨어집니다.


In [ ]:
NVBANDWIDTH_RESULT = run(
    [
        "nvbandwidth",
        "-t",
        "host_to_device_memcpy_ce",
        "device_to_host_memcpy_ce",
        "host_to_device_memcpy_sm",
        "device_to_host_memcpy_sm",
    ],
    timeout=900,
)

GPU_RESULT_PATH = WORK_DIR / f"gpu_results_{PROFILE_RUN_ID}.json"
GPU_RESULT_PATH.write_text(
    json.dumps(
        {
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "elements": ELEMENTS,
            "managed_modes": MANAGED_MODES,
            "profile_targets": PROFILE_TARGETS,
            "program_results": MEMORY_RESULTS,
            "oversubscription": {
                "ready": OVERSUBSCRIPTION_READY,
                "hbm_free_bytes": HBM_FREE,
                "hbm_total_bytes": HBM_TOTAL,
                "system_available_bytes": SYSTEM_AVAILABLE,
                "elements": BIG_ELEMENTS,
                "total_bytes": BIG_TOTAL_BYTES,
                "results": OVERSUBSCRIPTION_RESULTS,
            },
            "nsys_reports": {name: str(path.relative_to(REPO_ROOT)) for name, path in NSYS_REPORTS.items()},
            "nvbandwidth_stdout": NVBANDWIDTH_RESULT.stdout,
        },
        ensure_ascii=False,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)
print(f"결과 저장: {GPU_RESULT_PATH.relative_to(REPO_ROOT)}")


## 8. 실행 결과 자동 확인 — 1분

다음 셀은 선택한 데이터 크기와 실행 모드, 각 프로그램의 `PASS`·`SKIP`, 시스템 할당 메모리 경로, 생성된 Nsight Systems 보고서와 JSON 결과 파일을 자동으로 요약합니다.


In [ ]:
program_status = {}
for name, output in MEMORY_RESULTS.items():
    if "result=PASS" in output:
        program_status[name] = "PASS"
    elif "result=SKIP" in output:
        program_status[name] = "SKIP"
    else:
        program_status[name] = "CHECK"

system_memory_path = (
    system_path
    if SYSTEM_MEMORY_SUPPORTED
    else "SKIP (GPU 직접 접근 기능 미지원)"
)
program_coverage_ready = set(program_status) == set(PROGRAM_COMMANDS)
required_programs_ready = all(
    program_status.get(name) == "PASS"
    for name in PROGRAM_COMMANDS
    if name != "system"
)
system_result_ready = program_status.get("system") in {"PASS", "SKIP"}
reports_ready = (
    set(NSYS_REPORTS) == set(PROFILE_TARGETS)
    and all(path.is_file() for path in NSYS_REPORTS.values())
)
result_file_ready = GPU_RESULT_PATH.is_file()
overall_ready = (
    program_coverage_ready
    and required_programs_ready
    and system_result_ready
    and reports_ready
    and result_file_ready
)

print("=== GPU 실습 자동 요약 ===")
print(f"원소 수              : {ELEMENTS:,}")
print(f"통합 메모리 모드     : {', '.join(MANAGED_MODES)}")
print(f"프로파일 대상        : {', '.join(PROFILE_TARGETS)}")
print("\n프로그램 실행 상태")
for name in PROGRAM_COMMANDS:
    print(f"  {name:18s} {program_status.get(name, 'CHECK')}")
print(f"\n시스템 할당 메모리 경로: {system_memory_path}")
print("\nHBM 초과 실습")
if OVERSUBSCRIPTION_READY:
    print(f"  요청 크기          : {BIG_TOTAL_BYTES / 2**30:.1f} GiB "
          f"(HBM 여유 {HBM_FREE / 2**30:.1f} GiB의 "
          f"{BIG_TOTAL_BYTES / HBM_FREE:.2f}배)")
    for name, record in OVERSUBSCRIPTION_RESULTS.items():
        text = record["output"]
        mark = next(
            (m for m in ("result=PASS", "result=OOM", "result=HOST_OOM")
             if m in text),
            "result=CHECK",
        )
        print(f"  {name:16s} {mark.split('=')[1]:9s} {record['seconds']:6.1f} s")
else:
    print("  SKIP (시스템 메모리 여유 부족)")
print("Nsight Systems 보고서")
for name, path in NSYS_REPORTS.items():
    print(f"  {name:18s} {path.relative_to(REPO_ROOT)}")
print(f"JSON 결과 파일       : {GPU_RESULT_PATH.relative_to(REPO_ROOT)}")
print(f"최종 상태            : {'PASS' if overall_ready else 'CHECK'}")


## 참고 — 사전 검증에서 관측한 값

강사가 행사 전 KISTI PILOT GH200 한 대에서 측정한 값입니다. **정확히 같을 필요는 없습니다.** 자릿수가 크게 다르면 강사에게 알립니다.

| 측정 항목 | 관측값 |
|---|---|
| `explicit` / `managed-demand` / `managed-prefetch` 결과 | 모두 `PASS` |
| 시스템 할당 메모리 경로 | (행사 전 기입: `ATS` / `HMM` / `SKIP`) |
| `host_to_device_memcpy_ce` | (행사 전 기입) GB/s |
| `device_to_host_memcpy_ce` | (행사 전 기입) GB/s |
| `host_to_device_memcpy_sm` | (행사 전 기입) GB/s |
| `device_to_host_memcpy_sm` | (행사 전 기입) GB/s |
| HBM 초과 요청 크기 | (행사 전 기입) GiB (HBM 여유의 __배) |
| HBM 초과 — `explicit` | `OOM` (예상된 결과) |
| HBM 초과 — `managed-demand` | (행사 전 기입) PASS, __ 초 |
| HBM 초과 — `system` | (행사 전 기입) PASS, __ 초 |
| 이 노트북 전체 실행 시간 | (행사 전 기입) 분 |

## 완료 확인

- [ ] 세 소스에서 메모리 할당·이동 API를 직접 확인했습니다.
- [ ] 데이터 크기와 통합 메모리 실행 모드를 직접 선택했습니다.
- [ ] 지원되는 실행의 계산 결과가 모두 `PASS`인지 확인했습니다.
- [ ] 두 개 이상의 프로파일 대상을 고르고 `.nsys-rep`를 만들었습니다.
- [ ] CUDA API·GPU 커널·메모리 활동의 순서를 설명할 수 있습니다.
- [ ] Copy Engine(`_ce`)과 SM 커널(`_sm`) 복사의 측정값 차이를 확인했습니다.
- [ ] HBM보다 큰 배열에서 명시적 복사가 실패하고 통합 메모리·시스템 할당 메모리는 동작하는 것을 확인했습니다.
- [ ] 그 차이를 만드는 것이 NVLink-C2C의 대역폭과 ATS라는 점을 설명할 수 있습니다.

## 이 결과의 해석 범위

- 프로파일러에는 오버헤드가 있습니다. `.nsys-rep`의 시간을 프로그램의 원래 실행 시간으로 읽지 않습니다.
- nvbandwidth는 선택한 복사 경로의 대역폭이고, 프로그램 전체 시간이나 커널 처리율과는 다른 지표입니다.
- GH200에서 요구 시 접근(`demand`)이 항상 느린 것은 아닙니다. 데이터 크기와 접근 패턴에 따라 직접 접근이 선택될 수 있으므로 추측하지 말고 Nsight 기록으로 판단합니다.
- 측정값은 지금 이 노드 상태의 관찰값입니다. 다른 시스템의 성능으로 옮겨 적지 않습니다.
- 7절의 PCIe 비교표는 이 노드에서 측정한 값이 아니라 각 링크의 일반적으로 알려진 이론값입니다. 자릿수 비교용으로만 사용합니다.
- HBM 초과 실습의 실행 시간은 한 노드를 함께 쓰는 다른 참가자의 메모리 사용량에 영향을 받습니다.

다음 세션은 [PhysicsNeMo 발사체 운동 PINN](../02_PhysicsNeMo/01_Projectile_PINN.ipynb)입니다.

---

## 출처와 라이선스

이 실습은 KSC 2026을 위해 별도로 작성한 소스 코드를 사용합니다. CUDA, Nsight Systems와 nvbandwidth에는 각 원저작물의 라이선스와 고지가 적용됩니다.
